Install Ollama

It seems that the `ollama` installation failed because `zstd` was not present, which is required for extraction. I will install `zstd` and then re-attempt the `ollama` installation and server startup.

In [4]:
# Install zstd
!sudo apt-get update && sudo apt-get install -y zstd

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [355 B]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [101 kB]
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,840 kB]
Get:7 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:8 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [3,119 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-sec

Now that `zstd` is installed, I will re-run the Ollama installation.

In [5]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


With Ollama hopefully installed, I will now attempt to start the Ollama server.

Start Ollama Server

In [36]:
import subprocess
import time

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(5)

print("✅ Ollama server started!")

✅ Ollama server started!


Check Ollama

In [37]:
!ollama --version

ollama version is 0.32.1


Download the model

In [38]:
!ollama pull llama3.1:8b

Verify the model

In [39]:
!ollama list

NAME           ID              SIZE      MODIFIED      
llama3.1:8b    46e0c10c039e    4.9 GB    3 seconds ago    


Install Project Libraries

In [40]:
!pip install -q \
langchain \
langgraph \
langchain-community \
langchain-ollama \
duckduckgo-search \
ddgs \
reportlab \
graphviz \
grandalf

Test Ollama for python

In [41]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.1:8b",
    temperature=0.3
)

response = llm.invoke("Say hello in one sentence.")

print(response.content)

Hello, how are you today?


In [12]:
import os

os.makedirs("outputs", exist_ok=True)

print("Output folder created.")

Output folder created.


In [13]:
%%writefile llm.py

from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.1:8b",
    temperature=0.3
)

Writing llm.py


In [20]:
%%writefile state.py
"""
state.py

This file defines the shared state used by all agents.
Each agent reads from and updates this state.
"""

from typing import TypedDict, List


class ResearchState(TypedDict):
    """
    Shared state passed between all LangGraph nodes.
    """

    # Topic provided by the user
    topic: str

    # Questions generated by the planner
    questions: List[str]

    #ask for human approval
    approved: bool

    # Research findings from DuckDuckGo
    research_notes: List[str]

    # Draft report written by Writer
    draft_report: str

    # Final report after FactChecker
    final_report: str

    # URLs collected during research (bonus citations)
    sources: List[str]

Writing state.py


In [28]:
%%writefile prompts.py
"""
prompts.py

All prompts used by the agents.
"""

PLANNER_PROMPT = """
You are an expert research planner.

Research Topic:
{topic}

Generate exactly four research questions that cover the topic comprehensively.

Return only the questions.
"""
RESEARCH_SUMMARY_PROMPT = """
You are an expert research analyst.

Research Question:
{question}

Below are the DuckDuckGo search results.

Read them carefully.

Extract the most important facts.

Write one concise paragraph summarizing the findings.

Search Results:

{research}
"""
WRITER_PROMPT = """
You are a professional research report writer.

Research Topic:
{topic}

Research Notes:
{research}

Source URLs:
{sources}

Write a detailed research report of approximately 1000 words.

Use proper headings.

At the end of the report, add:

## References

Include every source URL exactly as provided.

Do not invent URLs.

Only include URLs provided above.
"""

FACTCHECK_PROMPT = """
You are an expert fact checker.

Review the report below.

Correct factual mistakes.
Improve clarity.
Remove repetition.
Keep headings.

Report:

{report}
"""

Writing prompts.py


In [27]:
%%writefile agents.py
"""
agents.py

This file contains all AI agents used in the LangGraph workflow.

Agents:
1. Planner      -> Breaks the topic into research questions.
2. Researcher   -> Searches the web and summarizes findings.
3. Writer       -> Generates the research report.
4. Fact Checker -> Reviews and improves the report.
"""

# DuckDuckGo search tool for web research
from langchain_community.tools import DuckDuckGoSearchResults

# Import shared workflow state
from state import ResearchState

# Import the LLM initialized in llm.py
from llm import llm

# Import prompts from prompts.py
from prompts import (
    PLANNER_PROMPT,
    RESEARCH_SUMMARY_PROMPT,
    WRITER_PROMPT,
    FACTCHECK_PROMPT,
)

# ---------------------------------------------------
# Initialize DuckDuckGo Search Tool
# ---------------------------------------------------

# output_format="list" returns search results as a list of dictionaries
# Each dictionary contains title, snippet, and URL.
search_tool = DuckDuckGoSearchResults(
    output_format="list"
)

# ===================================================
# Planner Agent
# ===================================================

print("\n========== Planner Agent ==========")

def planner(state: ResearchState):
    """
    Planner Agent

    Reads the topic from the shared state
    and generates four research questions.
    """

    # Read the research topic from the shared state
    topic = state["topic"]

    # Insert the topic into the planner prompt
    prompt = PLANNER_PROMPT.format(
        topic=topic
    )

    # Send prompt to the LLM
    response = llm.invoke(prompt)

    # Convert the response into a Python list
    # Each line becomes one research question
    questions = [
        line.strip("- ").strip()
        for line in response.content.split("\n")
        if line.strip()
    ]

    # Return updated state
    return {
        "questions": questions
    }

# ===================================================
# Human Approval Agent
# ===================================================

print("\n========== Human Approval ==========")

def human_approval(state: ResearchState):
    """
    Human Approval Agent

    Displays the generated research questions and asks
    the user whether to continue.
    """

    print("\n========== Human Approval ==========\n")

    print("Generated Research Questions:\n")

    for i, question in enumerate(state["questions"], start=1):
        print(f"{i}. {question}")

    choice = input("\nContinue with research? (yes/no): ")

    approved = choice.strip().lower() == "yes"

    return {
        "approved": approved
    }

def approval_router(state: ResearchState):

    if state["approved"]:
        return "approved"

    return "rejected"

# ===================================================
# Researcher Agent
# ===================================================

print("\n========== Research Agent ==========")

def researcher(state: ResearchState):
    """
    Research Agent

    Searches DuckDuckGo for each research question
    and summarizes the search results.
    """

    print("\n========== Research Agent ==========\n")

    research_notes = []
    sources = []

    # Loop through every research question
    for question in state["questions"]:

        print(f"Researching: {question}")

        # Perform web search
        search_results = search_tool.invoke(question)

        # Save raw search results as a source
        sources.append(str(search_results))

        # Ask the LLM to summarize the search results
        prompt = RESEARCH_SUMMARY_PROMPT.format(
            question=question,
            research=search_results
        )

        summary = llm.invoke(prompt)

        research_notes.append(summary.content)

    return {
        "research_notes": research_notes,
        "sources": sources
    }


# ===================================================
# Writer Agent
# ===================================================

print("\n========== Writer Agent ==========")

def writer(state: ResearchState):
    """
    Writer Agent

    Combines all summarized research
    and generates a complete report.
    """

    # Merge all research notes into one document
    research = "\n\n".join(state["research_notes"])

    # Combine all collected URLs into a single string
    sources = "\n".join(list(dict.fromkeys(state["sources"])))

    prompt = WRITER_PROMPT.format(
    topic=state["topic"],
    research=research,
    sources=sources
)

    # Generate the report
    response = llm.invoke(prompt)

    # Save the draft report
    return {
        "draft_report": response.content
    }


# ===================================================
# Fact Checker Agent
# ===================================================

print("\n========== Fact Checker Agent ==========")

def fact_checker(state: ResearchState):
    """
    Fact Checker Agent

    Reviews the draft report,
    improves clarity,
    removes repetition,
    and checks factual consistency.
    """

    # Fill the fact-checking prompt
    prompt = FACTCHECK_PROMPT.format(
        report=state["draft_report"]
    )

    # Generate improved report
    response = llm.invoke(prompt)

    # Save final report
    return {
        "final_report": response.content
    }

Writing agents.py


In [23]:
%%writefile graph.py
"""
graph.py

This file creates the LangGraph workflow by connecting
all AI agents in the correct execution order.
"""

# Import StateGraph and special START/END nodes
from langgraph.graph import StateGraph, START, END

# Import the shared state
from state import ResearchState

# Import all agent functions
from agents import (
    planner,
    human_approval,
    approval_router,
    researcher,
    writer,
    fact_checker
)

# ===================================================
# Create the Workflow
# ===================================================

# Create a new graph using our shared state
workflow = StateGraph(ResearchState)

# ===================================================
# Add Nodes (Agents)
# ===================================================

# Every agent becomes one node in the graph

workflow.add_node("Planner", planner)

workflow.add_node("HumanApproval", human_approval)

workflow.add_node("Researcher", researcher)

workflow.add_node("Writer", writer)

workflow.add_node("FactChecker", fact_checker)

# ===================================================
# Connect the Nodes
# ===================================================

# Define the order in which agents execute

workflow.add_edge(START, "Planner")

workflow.add_edge("Planner", "HumanApproval")

workflow.add_conditional_edges(

    "HumanApproval",

    approval_router,

    {
        "approved": "Researcher",

        "rejected": END
    }

)

workflow.add_edge("Researcher", "Writer")

workflow.add_edge("Writer", "FactChecker")

workflow.add_edge("FactChecker", END)

# ===================================================
# Compile the Graph
# ===================================================

# Compile converts the workflow into an executable graph
graph = workflow.compile()

# ===================================================
# Save Workflow Diagram
# ===================================================

try:
    png = graph.get_graph().draw_mermaid_png()

    with open("workflow.png", "wb") as f:
        f.write(png)

    print("✅ Workflow diagram saved as workflow.png")

except Exception as e:
    print("Could not generate workflow diagram.")
    print(e)

Writing graph.py


In [24]:
%%writefile utils.py
"""
utils.py

Utility functions used by the project.
"""

from reportlab.platypus import SimpleDocTemplate, Paragraph
from reportlab.lib.styles import getSampleStyleSheet


def save_pdf(report, filename="outputs/report.pdf"):
    """
    Save the final report as a PDF file.
    """

    doc = SimpleDocTemplate(filename)

    styles = getSampleStyleSheet()

    story = []

    story.append(
        Paragraph("<b>Research Report</b>", styles["Heading1"])
    )

    story.append(
        Paragraph(report.replace("\n", "<br/>"), styles["BodyText"])
    )

    doc.build(story)

Writing utils.py


In [31]:
%%writefile main.py
"""
main.py

Entry point of the Multi-Agent Research Company.

Workflow:
User Input
      ↓
Planner
      ↓
Human Approval
      ↓
Researcher
      ↓
Writer
      ↓
Fact Checker
      ↓
Save Report (.md and .pdf)
"""

import os

from graph import graph
from utils import save_pdf


# ===================================================
# Main Function
# ===================================================

def main():

    print("=" * 60)
    print("📝 Multi-Agent Research Company")
    print("=" * 60)

    # ---------------------------------------------
    # Get research topic
    # ---------------------------------------------

    topic = input("\nEnter a research topic: ")

    # ---------------------------------------------
    # Initial Shared State
    # ---------------------------------------------

    initial_state = {
        "topic": topic,
        "questions": [],
        "approved": False,
        "research_notes": [],
        "draft_report": "",
        "final_report": "",
        "sources": []
    }

    print("\n🚀 Starting Research Workflow...\n")

    # ---------------------------------------------
    # Run LangGraph
    # ---------------------------------------------

    final_state = graph.invoke(initial_state)

    # ---------------------------------------------
    # Check if user rejected the workflow
    # ---------------------------------------------

    if not final_state["approved"]:
        print("\n❌ Research cancelled by the user.")
        return

    # ---------------------------------------------
    # Create outputs folder
    # ---------------------------------------------

    os.makedirs("outputs", exist_ok=True)

    # ---------------------------------------------
    # Save Markdown Report
    # ---------------------------------------------

    report_path = "outputs/report.md"

    with open(report_path, "w", encoding="utf-8") as file:

        file.write("# Research Report\n\n")

        # Write final report
        file.write(final_state["final_report"])

        # Add References
        file.write("\n\n---\n")
        file.write("## References\n\n")

        # Remove duplicate URLs
        unique_sources = list(dict.fromkeys(final_state["sources"]))

        if unique_sources:
            for url in unique_sources:
                file.write(f"- {url}\n")
        else:
            file.write("No references available.\n")

    # ---------------------------------------------
    # Save PDF Report
    # ---------------------------------------------

    save_pdf(final_state["final_report"])

    # ---------------------------------------------
    # Display Result
    # ---------------------------------------------

    print("\n" + "=" * 60)
    print("✅ Research Completed Successfully!")
    print("=" * 60)

    print("\n📄 Markdown Report : outputs/report.md")
    print("📕 PDF Report      : outputs/report.pdf")

    print("\n" + "=" * 60)
    print("Report Preview")
    print("=" * 60)

    print(final_state["final_report"][:1000])

    print("\n...(Preview Truncated)...\n")


# ===================================================
# Run Program
# ===================================================

if __name__ == "__main__":
    main()

Writing main.py


In [42]:
!python main.py


========== Planner Agent ==========

========== Human Approval ==========

========== Research Agent ==========

========== Writer Agent ==========

========== Fact Checker Agent ==========
✅ Workflow diagram saved as workflow.png
📝 Multi-Agent Research Company

Enter a research topic: Prompt Engineering

🚀 Starting Research Workflow...


========== Human Approval ==========

Generated Research Questions:

1. 1. What are the key factors influencing the effectiveness of prompt engineering in natural language processing tasks, and how can they be optimized?
2. 2. How do different types of prompts (e.g., open-ended, multiple-choice, or ranking) impact model performance and user experience in various applications, such as chatbots or text classification?
3. 3. What are the implications of using pre-defined versus dynamically generated prompts on model adaptability, data efficiency, and interpretability in real-world scenarios?
4. 4. Can prompt engineering be used to mitigate potential bia